In [9]:
import pandas as pd

#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")

#Fill missing values in 'InventoryRatio' using the median
train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
test_df['InventoryRatio'].fillna(test_df['InventoryRatio'].median(), inplace=True)

#Display the first few rows
train_df.head()

<ipython-input-9-5e2c0f7fb993>:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
<ipython-input-9-5e2c0f7fb993>:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].met

,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,Sales
0,Q1,CMP01,5.44,0.05,-0.04,CCC,Buy,South,Metal Fabrication,2.02,1507
1,Q2,CMP01,5.77,0.03,0.00,CCC,Hold,South,Metal Fabrication,2.01,2968
2,Q3,CMP01,4.10,0.06,-0.02,BBB,Buy,South,Metal Fabrication,2.02,3007
3,Q4,CMP01,1.58,0.01,0.02,BBB,Buy,South,Metal Fabrication,1.98,1467
4,Q5,CMP01,2.56,-0.07,0.02,CCC,Buy,South,Metal Fabrication,1.96,2860


In [10]:
#Convert Quarter to numeric representation
train_df['Quarter'] = train_df['Quarter'].str.extract('(\d+)').astype(int)
test_df['Quarter'] = test_df['Quarter'].str.extract('(\d+)').astype(int)

#Convert Quarter to Month (approximation)
train_df['Month'] = train_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)
test_df['Month'] = test_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)

#Merge economic indicators based on Month
train_merged = train_df.merge(economic_indicators_df, on='Month', how='left')
test_merged = test_df.merge(economic_indicators_df, on='Month', how='left')

#Drop 'Month' after merging
train_merged.drop(columns=['Month'], inplace=True)
test_merged.drop(columns=['Month'], inplace=True)

#Display merged dataset
train_merged.head()

,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,Sales,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,1,CMP01,5.44,0.05,-0.04,CCC,Buy,South,Metal Fabrication,2.02,1507,67.2,1.5385,55.5,20847.8,57.083078,56.512247,54.628506,56.512247,57.083078
1,2,CMP01,5.77,0.03,0.00,CCC,Hold,South,Metal Fabrication,2.01,2968,65.2,2.7775,59.2,21315.8,48.503429,46.417782,46.417782,48.018395,43.653086
2,3,CMP01,4.10,0.06,-0.02,BBB,Buy,South,Metal Fabrication,2.02,3007,51.5,2.9635,52.2,21570.7,31.941453,32.676107,30.567971,31.622039,35.135599
3,4,CMP01,1.58,0.01,0.02,BBB,Buy,South,Metal Fabrication,1.98,1467,59.9,4.1780,50.4,21665.5,40.092490,39.691565,41.014617,41.014617,44.101739
4,5,CMP01,2.56,-0.07,0.02,CCC,Buy,South,Metal Fabrication,1.96,2860,64.9,3.6430,46.9,21659.7,53.220733,49.175957,50.932241,50.932241,53.220733


In [11]:
from sklearn.preprocessing import LabelEncoder

#Define categorical features to encode
categorical_features = ['Company', 'Region', 'Industry', 'Bond rating', 'Stock rating']

#Apply Label Encoding
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    train_merged[col] = le.fit_transform(train_merged[col])
    test_merged[col] = le.transform(test_merged[col])  # Use same encoding for test data
    label_encoders[col] = le  # Store encoders

#Display encoded dataset
train_merged.head()

,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,Sales,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,1,0,5.44,0.05,-0.04,6,0,2,2,2.02,1507,67.2,1.5385,55.5,20847.8,57.083078,56.512247,54.628506,56.512247,57.083078
1,2,0,5.77,0.03,0.00,6,1,2,2,2.01,2968,65.2,2.7775,59.2,21315.8,48.503429,46.417782,46.417782,48.018395,43.653086
2,3,0,4.10,0.06,-0.02,5,0,2,2,2.02,3007,51.5,2.9635,52.2,21570.7,31.941453,32.676107,30.567971,31.622039,35.135599
3,4,0,1.58,0.01,0.02,5,0,2,2,1.98,1467,59.9,4.1780,50.4,21665.5,40.092490,39.691565,41.014617,41.014617,44.101739
4,5,0,2.56,-0.07,0.02,6,0,2,2,1.96,2860,64.9,3.6430,46.9,21659.7,53.220733,49.175957,50.932241,50.932241,53.220733


In [12]:
#Interaction terms
train_merged["Revenue_Marketshare"] = train_merged["RevenueGrowth"] * train_merged["Marketshare"]
train_merged["Inventory_Quick"] = train_merged["InventoryRatio"] * train_merged["QuickRatio"]

test_merged["Revenue_Marketshare"] = test_merged["RevenueGrowth"] * test_merged["Marketshare"]
test_merged["Inventory_Quick"] = test_merged["InventoryRatio"] * test_merged["QuickRatio"]

#Rolling averages for economic indicators
rolling_features = ["Consumer Sentiment", "Interest Rate", "PMI", "Money Supply",
                    "NationalEAI", "EastEAI", "WestEAI", "SouthEAI", "NorthEAI"]

for feature in rolling_features:
    train_merged[f"{feature}_rolling"] = train_merged[feature].rolling(window=3, min_periods=1).mean()
    test_merged[f"{feature}_rolling"] = test_merged[feature].rolling(window=3, min_periods=1).mean()

#Display dataset with new features
train_merged.head()


,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,...,Inventory_Quick,Consumer Sentiment_rolling,Interest Rate_rolling,PMI_rolling,Money Supply_rolling,NationalEAI_rolling,EastEAI_rolling,WestEAI_rolling,SouthEAI_rolling,NorthEAI_rolling
0,1,0,5.44,0.05,-0.04,6,0,2,2,2.02,...,10.9888,67.200000,1.538500,55.500000,20847.800000,57.083078,56.512247,54.628506,56.512247,57.083078
1,2,0,5.77,0.03,0.00,6,1,2,2,2.01,...,11.5977,66.200000,2.158000,57.350000,21081.800000,52.793253,51.465014,50.523144,52.265321,50.368082
2,3,0,4.10,0.06,-0.02,5,0,2,2,2.02,...,8.2820,61.300000,2.426500,55.633333,21244.766667,45.842653,45.202045,43.871419,45.384227,45.290588
3,4,0,1.58,0.01,0.02,5,0,2,2,1.98,...,3.1284,58.866667,3.306333,53.933333,21517.333333,40.179124,39.595151,39.333457,40.218350,40.963475
4,5,0,2.56,-0.07,0.02,6,0,2,2,1.96,...,5.0176,58.766667,3.594833,49.833333,21631.966667,41.751559,40.514543,40.838277,41.189633,44.152690


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

#Define features and target variable
X = train_merged.drop(columns=['Sales'])  # Features
y = train_merged['Sales']  # Target variable

#Split data into training and validation sets (80-20 split)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

#Predict on validation set
y_pred = rf_model.predict(X_val)

#Evaluate performance
mae = mean_absolute_error(y_val, y_pred)
print(f"Baseline Random Forest MAE: {mae:.2f}")

Baseline Random Forest MAE: 1380.27


In [15]:
from sklearn.model_selection import RandomizedSearchCV

#Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

#Initialize Randomized Search
random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                                   param_grid, n_iter=10, cv=3, scoring='neg_mean_absolute_error',
                                   random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

#Best model from tuning
best_rf_model = random_search.best_estimator_

#Predict on validation set with tuned model
y_val_pred_tuned = best_rf_model.predict(X_val)

#Evaluate tuned model
tuned_mae = mean_absolute_error(y_val, y_val_pred_tuned)
print(f"Tuned Random Forest MAE: {tuned_mae:.2f}")


Tuned Random Forest MAE: 1375.82


In [16]:
#Predict sales for test data
test_predictions = best_rf_model.predict(test_merged.drop(columns=['RowID']))

#Prepare submission file
submission = pd.DataFrame({'ID': test_df['RowID'], 'Sales': test_predictions})
submission.to_csv("newtunedrandomforestsubmission.csv", index=False)

print("Submission file saved as 'newtunedrandomforestsubmission.csv'.")


Submission file saved as 'newtunedrandomforestsubmission.csv'.
